In [1]:
import os
from pathlib import Path
import json
import datetime
import requests
import pandas as pd
from zoneinfo import ZoneInfo

pw_file = Path.cwd().parent.parent / ".json"
with open(pw_file, encoding="utf-8") as f:
    my_file = json.load(f)
API_KEY = my_file["SOLAR_EDGE_API_KEY"]
SITE_ID = my_file["SOLAR_EDGE_SITE_ID"]

BASE_URL = f"https://monitoringapi.solaredge.com/site/{SITE_ID}/powerDetails"

START_DATE = datetime.datetime(2025, 1, 1)
END_DATE = datetime.datetime.now()

METERS = "Production,Consumption,SelfConsumption,FeedIn,Purchased"

# =========================
# Helper: Monatsweise Zeiträume erzeugen
# =========================
def month_ranges(start, end):
    current = start
    while current <= end:
        next_month = (current.replace(day=1) + datetime.timedelta(days=32)).replace(day=1)
        month_end = min(next_month - datetime.timedelta(seconds=1), end)
        yield current, month_end
        current = next_month

# =========================
# API-Abfragen & Records sammeln
# =========================
records = []

for start, end in month_ranges(START_DATE, END_DATE):

    params = {
        "startTime": start.strftime("%Y-%m-%d %H:%M:%S"),
        "endTime": end.strftime("%Y-%m-%d %H:%M:%S"),
        "meters": METERS,
        "api_key": API_KEY
    }

    print(f"Fetching {params['startTime']} → {params['endTime']}")

    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()

    data = response.json()["powerDetails"]

    for meter in data["meters"]:
        meter_type = meter["type"]
        for v in meter["values"]:
            records.append({
                "date": v["date"],
                "type": meter_type,
                "value": v.get("value", 0)
            })

# =========================
# DataFrame bauen
# =========================
df = pd.DataFrame(records)

# Solar Edge liefert die eigentlich die lokalen Zeiten. Es gibt aber keine Zeitumstellung.
# Daher zuerst mit +01:00 lokalisieren und dann nach Europe/Vienna konvertieren.
df["date"] = (pd.to_datetime(df["date"]).dt.tz_localize("+01:00")).dt.tz_convert(ZoneInfo("Europe/Vienna"))
df = df.dropna(subset=["date"])

df = (
    df.pivot_table(
        index="date",
        columns="type",
        values="value",
        aggfunc="sum"
    )
    .reset_index()
    .sort_values("date")
)

# =========================
# CSV speichern
# =========================
df.to_csv("energy_data.csv", index=False)

print(df.head())
print(df.tail())


Fetching 2025-01-01 00:00:00 → 2025-01-31 23:59:59
Fetching 2025-02-01 00:00:00 → 2025-02-28 23:59:59
Fetching 2025-03-01 00:00:00 → 2025-03-31 23:59:59
Fetching 2025-04-01 00:00:00 → 2025-04-30 23:59:59
Fetching 2025-05-01 00:00:00 → 2025-05-31 23:59:59
Fetching 2025-06-01 00:00:00 → 2025-06-30 23:59:59
Fetching 2025-07-01 00:00:00 → 2025-07-31 23:59:59
Fetching 2025-08-01 00:00:00 → 2025-08-31 23:59:59
Fetching 2025-09-01 00:00:00 → 2025-09-30 23:59:59
Fetching 2025-10-01 00:00:00 → 2025-10-31 23:59:59
Fetching 2025-11-01 00:00:00 → 2025-11-30 23:59:59
Fetching 2025-12-01 00:00:00 → 2025-12-31 23:59:59
Fetching 2026-01-01 00:00:00 → 2026-01-31 23:59:59
Fetching 2026-02-01 00:00:00 → 2026-02-22 17:07:46
type                      date  Consumption  FeedIn  Production  Purchased  \
0    2025-01-01 00:00:00+01:00    126.95957     0.0         0.0  126.95957   
1    2025-01-01 00:15:00+01:00    155.28748     0.0         0.0  155.28748   
2    2025-01-01 00:30:00+01:00    128.95338     0.0 